## Example target

We reproduce this contraction:

`np.einsum('ij,jk->ik', A, B)`

which is matrix multiplication.

In [4]:
import numpy as np


def einsum_matmul(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    """Reference using NumPy einsum: 'ij,jk->ik'."""
    return np.einsum("ij,jk->ik", A, B)


def loop_matmul_from_vectors(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    """
    Manual implementation of 'ij,jk->ik' using for-loops after
    flattening both operands into 1D vectors.
    """
    A = np.asarray(A)
    B = np.asarray(B)

    if A.ndim != 2 or B.ndim != 2:
        raise ValueError("A and B must both be 2D arrays")

    m, n = A.shape
    n2, p = B.shape
    if n != n2:
        raise ValueError("Inner dimensions must match")

    # Convert both matrices to vectors first
    A_vec = A.reshape(-1)
    B_vec = B.reshape(-1)

    # Output matrix
    C = np.zeros((m, p), dtype=np.result_type(A.dtype, B.dtype))

    # C[i, k] = sum_j A[i, j] * B[j, k]
    # A[i, j] in A_vec is at i*n + j
    # B[j, k] in B_vec is at j*p + k
    for i in range(m):
        for k in range(p):
            total = 0.0
            for j in range(n):
                total += A_vec[i * n + j] * B_vec[j * p + k]
            C[i, k] = total

    return C

In [5]:
# Demo data
rng = np.random.default_rng(42)
A = rng.normal(size=(3, 4))
B = rng.normal(size=(4, 2))

C_einsum = einsum_matmul(A, B)
C_loops = loop_matmul_from_vectors(A, B)

print("einsum result:", C_einsum)
print("loop result:", C_loops)
print("close?", np.allclose(C_einsum, C_loops, atol=1e-10))

einsum result: [[ 0.63688722  0.47058701]
 [-0.96827145 -1.18712946]
 [ 0.60761486 -0.16799613]]
loop result: [[ 0.63688722  0.47058701]
 [-0.96827145 -1.18712946]
 [ 0.60761486 -0.16799613]]
close? True
